In [31]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})

In [32]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [33]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [34]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())


Missing after unmasking:
age      2
spend    1
dtype: int64


In [35]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())


Missing after imputing: 0


In [36]:
# -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [37]:
# -----------------------------------------------------------
# 🔹 3B. FIX DATA TYPES
# -----------------------------------------------------------

# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)


id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [38]:
# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0


In [39]:
# -----------------------------------------------------------
# 🔹 4B. ONE WAY TO TREAT THEM — CAP (winsorise)
# -----------------------------------------------------------

# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


In [40]:
# -----------------------------------------------------------
# 🔹 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
# -----------------------------------------------------------

print(df['city'].value_counts())

city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64


In [41]:
# -----------------------------------------------------------
# 🔹 5B. STANDARDISE THE STRINGS
# -----------------------------------------------------------

s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())

city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [42]:
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')

# 1. strip + lower
# YOUR CODE HERE

# 2. map 'n.y.' -> 'new york'  (after lowering)
# YOUR CODE HERE

# 3. value_counts()
# YOUR CODE HERE

In [43]:
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

Final shape: (7, 6)
Missing values: 0
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,120.5,2024-01-08
4,5,Eve,29.5,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
